In [1]:
from sentinelhub import BBox, CRS, SHConfig, bbox_to_dimensions

# 🔐 API config
config = SHConfig()
config.sh_client_id = "ca118fee-7d8c-4806-b8f6-b92bc64ea610"
config.sh_client_secret = "ZMnmBhby8LV6d8WinTNWiWrMJ6ypAyAI"

# 📍 Bhopal approx bounding box
bbox = BBox(
    bbox=[77.30, 23.20, 77.50, 23.35],  # [min_lon, min_lat, max_lon, max_lat]
    crs=CRS.WGS84
)

# 📏 Image size calculation
resolution = 60  # meters
size = bbox_to_dimensions(bbox, resolution=resolution)
print(f"Image Size (Width, Height): {size}")

Image Size (Width, Height): (337, 283)


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sentinelhub import SentinelHubRequest, DataCollection, MimeType

# 🎨 True Color (RGB) Evalscript
evalscript_true_color = """
//VERSION=3
function setup() {
  return {
    input: ["B02", "B03", "B04"],
    output: { bands: 3 }
  };
}

function evaluatePixel(sample) {
  // Multiply by 2.5 to increase brightness of the visual image
  return [2.5 * sample.B04, 2.5 * sample.B03, 2.5 * sample.B02];
}
"""

# 🚀 Request for True Color
request_rgb = SentinelHubRequest(
    evalscript=evalscript_true_color,
    input_data=[
        SentinelHubRequest.input_data(
            data_collection=DataCollection.SENTINEL2_L2A,
            time_interval=("2024-01-01", "2024-12-31"),
        )
    ],
    responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
    bbox=bbox,
    size=size,
    config=config
)

# 📥 Get and Save Normal Image
rgb_image = request_rgb.get_data()[0]

# Values ko 0-1 range mein clip kar dete hain taaki colors sahi se render ho
rgb_image_clipped = np.clip(rgb_image, 0, 1)

plt.imsave("bhopal_normal_image.png", rgb_image_clipped)
print(f"Normal Image Shape: {rgb_image.shape}")
print("✅ Normal image saved as 'bhopal_normal_image.png'")

Normal Image Shape: (283, 337, 3)
✅ Normal image saved as 'bhopal_normal_image.png'


In [3]:
# 🧪 NDVI Evalscript
evalscript_ndvi = """
//VERSION=3
function setup() {
  return {
    input: ["B04", "B08"],
    output: { bands: 1 }
  };
}

function evaluatePixel(sample) {
  let ndvi = (sample.B08 - sample.B04) / (sample.B08 + sample.B04);
  return [ndvi];
}
"""

# 🚀 Request for NDVI
request_ndvi = SentinelHubRequest(
    evalscript=evalscript_ndvi,
    input_data=[
        SentinelHubRequest.input_data(
            data_collection=DataCollection.SENTINEL2_L2A,
            time_interval=("2024-01-01", "2024-12-31"),
        )
    ],
    responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
    bbox=bbox,
    size=size,
    config=config
)

# 📥 Get NDVI Image
ndvi_image = request_ndvi.get_data()[0]

# Shape ko (H, W, 1) se (H, W) mein convert karna zaroori hai matplotlib ke liye
ndvi_2d = np.squeeze(ndvi_image)

# NDVI normalize (0–1 range me convert)
ndvi_norm = (ndvi_2d - np.min(ndvi_2d)) / (np.max(ndvi_2d) - np.min(ndvi_2d))

# Save as PNG
plt.imsave("bhopal_ndvi_heatmap.png", ndvi_norm, cmap='RdYlGn')
print(f"NDVI Image Shape: {ndvi_2d.shape}")
print("✅ NDVI image saved as 'bhopal_ndvi_heatmap.png'")

NDVI Image Shape: (283, 337)
✅ NDVI image saved as 'bhopal_ndvi_heatmap.png'


In [4]:
# Grid logic and Greenery Calculation
green = np.sum(ndvi_2d > 0.3)
non_green = np.sum(ndvi_2d < 0.2)
total = ndvi_2d.size

print(f"Overall Green %: {(green/total)*100:.2f}%")
print(f"Overall Non-Green %: {(non_green/total)*100:.2f}%")
print("-" * 30)

rows, cols = ndvi_2d.shape
grid_size = 10
sector_h = rows // grid_size
sector_w = cols // grid_size

results = []

for i in range(grid_size):
    for j in range(grid_size):
        sector = ndvi_2d[
            i*sector_h:(i+1)*sector_h,
            j*sector_w:(j+1)*sector_w
        ]
        
        avg_ndvi = np.mean(sector)
        green_ratio = np.sum(sector > 0.3) / sector.size
        
        if avg_ndvi < 0.2:
            priority = "HIGH"
        elif avg_ndvi < 0.4:
            priority = "MEDIUM"
        else:
            priority = "LOW"
            
        area_factor = 100  # Adjust as needed
        trees_needed = int((1 - green_ratio) * area_factor)
        
        results.append({
            "sector": f"{i},{j}",
            "avg_ndvi": round(float(avg_ndvi), 3),
            "green_ratio": round(float(green_ratio), 2),
            "priority": priority,
            "trees_needed": trees_needed
        })

# Print top 5 high-priority sectors
high_priority = [r for r in results if r['priority'] == 'HIGH']
print("High Priority Sectors for Plantation:")
for r in high_priority[:5]:
    print(r)

Overall Green %: 88.93%
Overall Non-Green %: 11.07%
------------------------------
High Priority Sectors for Plantation:


In [5]:
import numpy as np
import matplotlib.pyplot as plt
from sentinelhub import SentinelHubRequest, DataCollection, MimeType

# 🎨 True Color (RGB) Evalscript with Alpha Mask
# Hum 4 bands le rahe hain: R, G, B aur ek Alpha (dataMask) band
evalscript_true_color = """
//VERSION=3
function setup() {
  return {
    input: ["B02", "B03", "B04", "dataMask"],
    output: { bands: 4 } 
  };
}

function evaluatePixel(sample) {
  // Raw values return kar rahe hain, normalization Python mein karenge
  return [sample.B04, sample.B03, sample.B02, sample.dataMask];
}
"""

# 🚀 Request for True Color
request_rgb = SentinelHubRequest(
    evalscript=evalscript_true_color,
    input_data=[
        SentinelHubRequest.input_data(
            data_collection=DataCollection.SENTINEL2_L2A,
            time_interval=("2024-01-01", "2024-12-31"),
        )
    ],
    responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
    bbox=bbox,
    size=size,
    config=config
)

rgba_data = request_rgb.get_data()[0]

# Split RGB and Alpha (Mask)
rgb_image = rgba_data[:, :, :3]
alpha_mask = rgba_data[:, :, 3]

# 🌟 Percentile Contrast Stretching
valid_pixels = rgb_image[alpha_mask > 0]

if len(valid_pixels) > 0:
    p2, p98 = np.percentile(valid_pixels, (2, 98))
    
    # FIX 1: Prevent division by zero if contrast is completely flat
    if p98 > p2:
        rgb_stretched = (rgb_image - p2) / (p98 - p2)
    else:
        rgb_stretched = rgb_image
else:
    # Fallback agar poori image khali hai
    rgb_stretched = rgb_image

# 🛠️ FIX 2: Safely handle any Infinity/NaNs and strictly clip to 0.0 - 1.0
rgb_stretched = np.nan_to_num(rgb_stretched)
rgb_stretched = np.clip(rgb_stretched, 0.0, 1.0)

alpha_mask = np.nan_to_num(alpha_mask)
alpha_mask = np.clip(alpha_mask, 0.0, 1.0)

# Re-attach the alpha mask
final_rgba_image = np.dstack((rgb_stretched, alpha_mask))

# 💾 Save the final enhanced image
plt.imsave("bhopal_normal_image_fixed.png", final_rgba_image)

print(f"Original RGB Min: {rgb_image.min():.3f}, Max: {rgb_image.max():.3f}")
print("✅ Fixed normal image saved as 'bhopal_normal_image_fixed.png'")

Original RGB Min: 5.000, Max: 255.000
✅ Fixed normal image saved as 'bhopal_normal_image_fixed.png'
